# Native-Sparse-Attention

author: [dhcode-cpp](https://github.com/dhcode-cpp)

blog: [【手撕NSA】DeepSeek新作-原生稀疏注意力-超长文(附代码)](https://zhuanlan.zhihu.com/p/24841366485)

1. Compress Attention
2. Selection Attention
3. Sliding Window Attenion
4. Gated Aggregation
5. Stride sletection attention


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
torch.manual_seed(42)

## Config

In [ ]:
t = 32 # token ids 
l = 8 # block
d = 8 # sliding stride
block_nums = t // l
dim = 16 # embedding dimension
heads = 4
head_dim = dim//heads
batch_size = 1

In [ ]:
X = torch.randn(batch_size, t, dim)

Wq = torch.randn(dim, dim)
Wk = torch.randn(dim, dim)
Wv = torch.randn(dim, dim)

Q = X @ Wq
K = X @ Wk
V = X @ Wv

# skip apply rope

In [ ]:
print(Q.shape)
print(K.shape)
print(V.shape)

torch.Size([1, 32, 16])
torch.Size([1, 32, 16])
torch.Size([1, 32, 16])


## Attention with different KV-len

## Token Compression

In [ ]:
d = 4
max_idx = round(( t - l ) / d)
print(max_idx)
print(torch.arange(max_idx) * d + 1)
print(torch.arange(max_idx) * d + l)

6
tensor([ 1,  5,  9, 13, 17, 21])
tensor([ 8, 12, 16, 20, 24, 28])


In [ ]:
d = l
max_idx = round(( t ) / d)
print(max_idx)
print(torch.arange(max_idx) * d + 1)
print(torch.arange(max_idx) * d + l)

4
tensor([ 1,  9, 17, 25])
tensor([ 8, 16, 24, 32])


In [ ]:
W_K_cmp = torch.randn(l, 1)
W_V_cmp = torch.randn(l, 1)
W_pe = torch.randn(t, dim)

In [ ]:
K_cmp = []
V_cmp = []
for i in range(max_idx):
    cur_K = K[:, i * d + 0: i * d + l , :] + W_pe[:l, :].unsqueeze(0)
    cur_V = V[:, i * d + 0: i * d + l , :] 
    cur_K = cur_K.transpose(1, 2) @ W_K_cmp 
    cur_V = cur_V.transpose(1, 2) @ W_V_cmp
    K_cmp.append(cur_K)
    V_cmp.append(cur_V)

K_cmp = torch.cat(K_cmp, dim = 2).transpose(1,2)
V_cmp = torch.cat(V_cmp, dim = 2).transpose(1,2)
print(K_cmp.shape)
print(V_cmp.shape)

torch.Size([1, 4, 16])
torch.Size([1, 4, 16])


In [ ]:
# multi-head attn
Q = Q + W_pe[:t, :].unsqueeze(0)
Q_mha = Q.view(1, t, heads, head_dim).transpose(1,2)
# Q_mha = Q_mha + W_pe.unsqueeze(1)

K_cmp_mha = K_cmp.view(1, block_nums, heads, head_dim).transpose(1,2)
V_cmp_mha = V_cmp.view(1, block_nums, heads, head_dim).transpose(1,2)
score_cmp = Q_mha @ K_cmp_mha.transpose(2,3) # bs, head, q_len, k_cmp_len
print(score_cmp.shape)

torch.Size([1, 4, 32, 4])


In [ ]:
p_cmp = F.softmax(score_cmp, dim = -1) 
o_cmp = p_cmp @ V_cmp_mha
print(o_cmp.shape)

o_cmp = o_cmp.transpose(2, 1).reshape(batch_size, t, dim)
print(o_cmp.shape)

torch.Size([1, 4, 32, 4])
torch.Size([1, 32, 16])


## Token Selection

In [ ]:
print(p_cmp.shape)
p_slc = p_cmp.sum(dim = 1)
print(p_slc.shape)

torch.Size([1, 4, 32, 4])
torch.Size([1, 32, 4])


In [ ]:
select_top_k = 2
value, idx = torch.topk(p_slc, dim = 2, k = select_top_k)
print(idx[0,0,:])
idx.shape

tensor([2, 1])


torch.Size([1, 32, 2])

In [ ]:
idx_slc_start = idx * d
idx_slc_end = idx * d + l
K_slc = torch.randn(batch_size, t, d * select_top_k, dim)
V_slc = torch.randn(batch_size, t, d * select_top_k, dim)
for i in range(batch_size):
    for j in range(t):
        for k in range(select_top_k):
            K_slc[i, j, k * d : k * d + l, :] = K[i, idx_slc_start[i, j, k ] :  idx_slc_end[i, j, k ] , :]
            V_slc[i, j, k * d : k * d + l, :] = V[i, idx_slc_start[i, j, k ] :  idx_slc_end[i, j, k ] , :]
print(K_slc.shape)
print(V_slc.shape)

torch.Size([1, 32, 16, 16])
torch.Size([1, 32, 16, 16])


In [ ]:
# shared head KV
# IN GQA Group: [1-head KV & N-head Q] ----repeat kv-head---> [N-head KV & N-head Q]

V_slc_mha = V_slc.view(batch_size, t, select_top_k * d, heads, head_dim).transpose(2,3)
V_slc = V_slc_mha.sum(dim = 2, keepdim = True)
print(V_slc.shape) # bs, seq_len, head, select_seq_len, head_dim

K_slc_mha = K_slc.view(batch_size, t, select_top_k * d, heads, head_dim).transpose(2,3)
K_slc = K_slc_mha.sum(dim = 2, keepdim = True)
print(V_slc.shape) # bs, seq_len, head, select_seq_len, head_dim

torch.Size([1, 32, 1, 16, 4])
torch.Size([1, 32, 1, 16, 4])


In [ ]:
# debug Q-1 and KV-16 attention
print(Q_mha.shape) # bs, head, seq, head_dim
print(Q_mha[:, :, 5, :].shape) # t=5
print(K_slc[:, 5, :, :, :].shape) # t=5

print(Q_mha[:, :, 5, :].unsqueeze(dim = 2).repeat(1, 1, select_top_k * d, 1).shape) # t=5
print(K_slc[:, 5, :, :, :].repeat(1, heads, 1, 1).shape) # t=5

Q_slc_j = Q_mha[:, :, 5, :].unsqueeze(dim = 2)
K_slc_j = K_slc[:, 5, :, :, :].repeat(1, heads, 1, 1)

attn_score_j = Q_slc_j @ K_slc_j.transpose(2,3)
print(attn_score_j.shape) # bs, head, seq_q, seq_slc_k

V_slc_j = V_slc[:, 5, :, :, :].repeat(1, heads, 1, 1)
print(V_slc_j.shape)

o_j = (attn_score_j @ V_slc_j).transpose(1,2).view(batch_size, 1, dim)
print(o_j.shape) # bs, j, dim

torch.Size([1, 4, 32, 4])
torch.Size([1, 4, 4])
torch.Size([1, 1, 16, 4])
torch.Size([1, 4, 16, 4])
torch.Size([1, 4, 16, 4])
torch.Size([1, 4, 1, 16])
torch.Size([1, 4, 16, 4])
torch.Size([1, 1, 16])


In [ ]:
o_slc = torch.zeros(batch_size, t, dim)
for j in range(t):
    Q_slc_j = Q_mha[:, :, j, :].unsqueeze(dim = 2)
    K_slc_j = K_slc[:, j, :, :, :].repeat(1, heads, 1, 1)
    V_slc_j = V_slc[:, j, :, :, :].repeat(1, heads, 1, 1)
    
    attn_score_j = Q_slc_j @ K_slc_j.transpose(2,3)
    p_slc_j = F.softmax(attn_score_j, dim = -1) 
    # print(p_slc.shape)

    o_slc_j = p_slc_j @ V_slc_j # bs, seq, dim   
    # print(o_slc_j.shape)

    o_slc_j = o_slc_j.transpose(1,2).view(batch_size, 1, dim)
    o_slc[:, j, :] = o_slc_j
    
print(o_slc.shape)

torch.Size([1, 32, 16])


### Token Selection details

1. NSA using GQA, so we have many group(>1)
2. every group has indipendent KV-Selection
3. In group, caluculative n-heads-Q and 1-heads-KV attention
4. In group 1-heads-kv repeat to n-heads-kv, but in NSA kernel, the 1-heads-kv send to SRAM shared memory. this procedure make less meomery asscess.

## sliding window attention

In [ ]:
# built sliding window attention
def get_window_mask(seq_len, window):
    mask = torch.ones(seq_len, seq_len, dtype = torch.long)
    mask = torch.tril(mask)
    win_mask = -torch.ones(seq_len - window, seq_len - window, dtype = torch.long)
    win_mask =  torch.tril(win_mask)
    mask[window:, :seq_len - window] += win_mask
    return mask
print(get_window_mask(7, 3)) # test
window_mask = get_window_mask(t, 8)

window_mask = 1 - window_mask
print(window_mask)

add_window_mask = window_mask * torch.tensor(float('-inf'))
add_window_mask = torch.nan_to_num(add_window_mask, nan=0.0)
print(add_window_mask)

tensor([[1, 0, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 0, 0, 0, 0],
        [0, 1, 1, 1, 0, 0, 0],
        [0, 0, 1, 1, 1, 0, 0],
        [0, 0, 0, 1, 1, 1, 0],
        [0, 0, 0, 0, 1, 1, 1]])
tensor([[0, 1, 1,  ..., 1, 1, 1],
        [0, 0, 1,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 0, 1, 1],
        [1, 1, 1,  ..., 0, 0, 1],
        [1, 1, 1,  ..., 0, 0, 0]])
tensor([[ 0.0000e+00, -3.4028e+38, -3.4028e+38,  ..., -3.4028e+38,
         -3.4028e+38, -3.4028e+38],
        [ 0.0000e+00,  0.0000e+00, -3.4028e+38,  ..., -3.4028e+38,
         -3.4028e+38, -3.4028e+38],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -3.4028e+38,
         -3.4028e+38, -3.4028e+38],
        ...,
        [-3.4028e+38, -3.4028e+38, -3.4028e+38,  ...,  0.0000e+00,
         -3.4028e+38, -3.4028e+38],
        [-3.4028e+38, -3.4028e+38, -3.4028e+38,  ...,  0.0000e+00,
          0.0000e+00, -3.4028e+38],
        [-3.4028e+38, -3.4028e+38, -

In [ ]:
# simplify multihead attention
S = Q @ K.transpose(1,2) / math.sqrt(dim)
S = S + add_window_mask # using add-style attention mask
S = F.softmax(S, dim = -1)
print(S)
o_win = S @ V
print(o_win.shape)

tensor([[[1.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00],
         [8.4573e-09, 1.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00],
         [1.0000e+00, 3.1539e-20, 7.9926e-12,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00],
         ...,
         [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 8.2928e-17,
          0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 3.2671e-01,
          1.0562e-02, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 9.5674e-08,
          2.6667e-11, 4.6384e-18]]])
torch.Size([1, 32, 16])


## Gated Aggregation

In [ ]:
W_gated = torch.randn(dim, 3) # 3: cmp, slc, win
gate = X @ W_gated
gate = F.sigmoid(gate)
print(gate.shape)

torch.Size([1, 32, 3])


In [ ]:
o_list = [o_cmp, o_slc, o_win]
o_star = torch.zeros(batch_size, t, dim)
for i in range(3):
    o_star += gate[:, :, i].unsqueeze(2) * o_list[i]
print(o_star.shape)

torch.Size([1, 32, 16])


## Appendix: stride sletection attention

$$
\mathbf{p}_t^\text{slc}[j] = \sum_{m=0}^{\frac{l'}{d}-1}\sum_{n=0}^{\frac{l}{d} -1} \mathbf{p}_t^\text{cmp}\left[\frac{l'}{d}j+m +n \right],
$$

In [ ]:
d = 8
t = 512 + d
l_cmp = 16

# case1. l_slc=
l_slc = 8 # from paper：“l‘ denote the selection block size”, or setting l_slc = {4, 8, 16, 32, ...}
m_max = l_slc // d
n_max = l_cmp // d
print(m_max)
print(n_max)

1
2


In [ ]:
# original is t token, compress -> t_cmp token.
# t_cmp = (t - d) // l_cmp
t_cmp = (t - l_cmp) // d
print(t_cmp)

63


In [ ]:
# t_cmp = (t - d) // l_cmp
t_cmp = (t - l_cmp) // d

p_cmp = torch.randn(t_cmp)
p_slc = torch.zeros_like(p_cmp)
j_factor = l_slc // d 

for j in range(t_cmp):
    for m in range(m_max):
        for n in range(n_max):
            idx = j_factor * j + m + n
            if idx >= t_cmp:
                continue
            else:
                p_slc[j] += p_cmp[idx]

print(p_slc)

tensor([ 1.5153, -1.2284, -1.1369,  1.2023, -0.5375,  0.2146,  1.4350,  1.5226,
         0.9039, -0.3535,  0.2453, -0.0667, -1.3982, -1.3786, -0.2436, -0.8676,
        -1.5858, -0.5830, -0.4754, -0.5481, -0.3486, -0.4307, -0.2940, -1.2607,
        -0.2955, -0.5889, -0.0674,  0.4382, -0.4188,  1.6259,  1.1821,  0.1836,
         0.8457,  0.0802, -1.3933, -1.9170, -0.3828,  0.5425,  0.3419, -0.1868,
        -1.6885, -0.6137,  0.2888, -1.1837, -1.6228, -1.2174, -1.2547, -0.5664,
         0.0350,  0.9818,  0.4365, -0.8080, -0.2830, -0.1650,  0.4615,  1.6315,
         1.0255,  0.4499, -0.7477, -1.2397, -0.4850,  1.1614,  1.9408])
